# Kafka Producer

In [9]:
!pip cache purge --quiet

In [10]:
!pip install confluent-kafka==2.11.0 --quiet

In [11]:
import json
import pandas as pd
import random
import time

from confluent_kafka import Producer, KafkaException
from datetime import datetime
from IPython.display import clear_output
from math import cos, radians

In [12]:
# Kafka config
conf = {
    "bootstrap.servers": "<bootstrap_server>",
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": "<api_key>",
    "sasl.password": "<api_secret>",
    "log_level": 0
}

topic = "iot-temperatures"

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [13]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [14]:
counter = 0
print_every = 10

# Query the database for min and max sensor IDs
query = "SELECT MIN(id) AS min_id, MAX(id) AS max_id FROM sensors;"
min_id, max_id = pd.read_sql(query, db_connection).iloc[0]

In [15]:
def generate_temperature(latitude: float) -> float:
    """
    Generate a temperature based on latitude:
    - Equator (lat ~ 0) -> hottest
    - Poles (lat ~ ±90) -> coldest
    Adds some random fluctuation.
    """
    # Base temperature: 30°C at equator, -10°C at poles
    base_temp = 30 * cos(radians(latitude)) - 10
    # Add small random variation ±5°C
    temp = base_temp + random.uniform(-5, 5)
    return round(temp, 2)

In [16]:
def delivery_report(err, msg):
    if err:
        print(f"Delivery failed: {err}")
    else:
        print(f"Delivered to {msg.topic()} [{msg.partition()}] @ offset {msg.offset()}")

In [17]:
def _produce_single_event():
    global counter
    # Pick a random sensor
    sensor_id = random.randint(min_id, max_id)

    # Query that specific sensor
    query = f"SELECT id, latitude FROM sensors WHERE id = {sensor_id};"
    sensor = pd.read_sql(query, db_connection).iloc[0]
    
    record = {
        "id": int(sensor["id"]),
        "temperature": generate_temperature(sensor["latitude"]),
        "timestamp": int(datetime.utcnow().timestamp() * 1000)
    }

    producer.produce(
        topic = topic,
        key = str(record["id"]),
        value = json.dumps(record),
        callback = delivery_report
    )

    # Immediately poll to trigger delivery callbacks
    producer.poll(0)

    counter += 1
    if counter % print_every == 0 or counter == 1:
        clear_output(wait = True)
        print(f"Produced {counter} records, latest: {record}")

In [18]:
def produce_events(num_messages = 20, interval = 0.2):
    try:
        if num_messages == -1:
            print("Producing events endlessly ...")
            while True:
                _produce_single_event()
                time.sleep(interval)
        else:
            for _ in range(num_messages):
                _produce_single_event()
                time.sleep(interval)
    except KeyboardInterrupt:
        print("Producer stopped by user.")
    finally:
        # Flush any remaining messages
        producer.flush(timeout = 10)
        print(f"Finished producing events (total {counter})")

In [20]:
producer = Producer(conf)
produce_events(num_messages = -1)

Produced 1210 records, latest: {'id': 64, 'temperature': 14.48, 'timestamp': 1783546505131}
Producer stopped by user.
Delivered to iot-temperatures [3] @ offset 368
Finished producing events (total 1210)
